In [24]:
import pandas as pd
import json
import re
from pathlib import Path
from pdf2image import convert_from_path
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials
from google.genai import errors as genai_errors
import mimetypes

# ==========================
# CONFIGURATION (COMPANY VM)
# ==========================

base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "PASTE_FRESH_TOKEN_HERE"   # <<< put fresh token here
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"   # or "models/gemini-2.5-pro" if your gateway needs that

# set this to True once to see the full raw JSON/text the model returns
DEBUG_SHOW_RAW = True

print("Connection Extractor Ready (company gateway).")


# ==========================
# PROMPT (CONNECTIONS + SEGMENTS)
# ==========================

def get_connection_prompt():
    return """
    You are an Engineering Wiring/Diagram Connection Extraction AI with strong
    2D spatial understanding.

    INPUT (ONE SHEET ONLY):
    - A single sheet of a technical drawing that may include:
      - Wiring harness diagrams
      - Electrical schematics
      - P&ID / instrumentation
      - Any diagram where components are connected by lines / wires / cables.

    The sheet may also include tables:
      - Connector tables
      - Pinout tables
      - Wire list / cable schedule
      - Length tables or scale information
      - Legends / symbol tables.

    YOUR TASK ON THIS SHEET:

    1. IDENTIFY CONNECTIONS
       - Follow wires/lines in the drawing (including bus lines).
       - A connection is:
             from_component.pin  --->  to_component.pin

       For each connection, extract:

       - from_component:
           Identifier / name of the source component or connector.
           Examples: "J1", "C101", "P2", "X1", "MOTOR1", "PLC_A".
       - from_pin:
           Pin or terminal number / name at the source.
           Examples: "1", "2", "A1", "L1", "+", "NO".
       - to_component:
           Identifier / name of the destination component or connector.
       - to_pin:
           Pin or terminal number / name at the destination.

       If exact pin labels are unclear:
         - Use the nearest clear label that corresponds to that terminal.
       If there is truly no visible pin label:
         - Set the pin field to null, but still return the connection if the
           two components are clearly linked by a wire/line.

    2. WIRE PROPERTIES FOR EACH CONNECTION (IF AVAILABLE)
       From any tables or labels on the drawing, extract:

       - wire_id:
           Wire or cable identifier.
           Examples: "W1", "Wire_01", "Cable-A", "Wire1.Red".
       - wire_gauge:
           Cross-section or gauge.
           Examples: "0.5 mm2", "1.5 mm²", "2.5 mm2", "20 AWG".
       - wire_color:
           Wire color or color code.
           Examples: "RED", "BLK", "Blue", "BK/RD".
       - core_count:
           For multi-core cables, number of cores (integer).

       If any field is not present or not readable, set that field to null.

    3. PATH SEGMENT DISTANCES (NO TOTAL LENGTH)
       Use your spatial reasoning to interpret the routing:

       - Follow the drawn route of each wire between its two endpoints.
       - Look for dimension labels written along that path, such as:
           "7 ft", "4 ft", "3 in", "150 mm", "2.5 m".
       - For each connection, build:
             segment_lengths = [segment_1, segment_2, ..., segment_n]
         where each segment_i is exactly one dimension label string including
         the unit text.

       Examples:
         ["7 ft", "7 ft", "1.5 in"]
         ["150 mm"]
         []

       If there are no usable distance labels for that connection:
           "segment_lengths": []

       Do NOT calculate or add any total length field.

    4. OUTPUT FORMAT (STRICT JSON, NO MARKDOWN)
       Return ONLY a valid JSON object exactly in this shape:

       {
         "connections": [
           {
             "from_component": "C101",
             "from_pin": "1",
             "to_component": "C102",
             "to_pin": "3",
             "wire_id": "Wire1.Red",
             "wire_gauge": "20 AWG",
             "wire_color": "Red",
             "core_count": 1,
             "segment_lengths": ["7 ft", "7 ft", "1.5 in"]
           }
         ]
       }

    RULES:
    - "connections" MUST always be present in the JSON.
    - Every item in "connections" MUST represent a real connection on this sheet.
    - Use null for fields you cannot determine (except segment_lengths which
      MUST be [] if unknown).
    - If there are absolutely no usable connections on the sheet:
         return exactly: { "connections": [] }.
    - Do NOT include any explanation, comments, or text outside the JSON.
    """


# ==========================
# MODEL CALL: ONE IMAGE → JSON
# ==========================

def analyze_image(image_path: str):
    """
    Analyze one page/image.
    Returns a dict: { "connections": [ ... ] }
    """
    print(f"   -> Analyzing page image: {image_path} ...")

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[get_connection_prompt(), image_part],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
            ),
        )

        raw_text = response.text or "{}"

        if DEBUG_SHOW_RAW:
            print("   RAW RESPONSE (first 600 chars):")
            print(raw_text[:600])

        # Strip code fences if present
        if "```json" in raw_text:
            raw_text = raw_text.split("```json")[1].split("```")[0]
        elif "```" in raw_text:
            raw_text = raw_text.split("```")[1].split("```")[0]

        data = json.loads(raw_text)

    except genai_errors.ClientError as e:
        print("      [ClientError] Gateway / auth / model issue.")
        print("      Message:", getattr(e, "message", e))
        resp = getattr(e, "response", None)
        if resp is not None and hasattr(resp, "text"):
            print("      Raw server response:")
            try:
                print(resp.text)
            except Exception:
                pass
        return {"connections": []}

    except Exception as e:
        print("      [Error] Could not analyze this page.")
        print("      Detail:", e)
        return {"connections": []}

    # Normalise structure
    if not isinstance(data, dict):
        data = {"connections": []}
    if "connections" not in data or not isinstance(data["connections"], list):
        data["connections"] = []

    return data


# ==========================
# PIPELINE: FILE (PDF or IMAGE) → EXCEL
# ==========================

def process_file(file_path: str):
    """
    - If PDF: convert each page to PNG (300 dpi).
    - If image: use directly.
    - For each page: extract connections + segment lengths.
    - Writes: <stem>_CONNECTIONS_SEGMENTS.xlsx (sheet 'Connections').
    """
    print(f"\n--- Processing File: {file_path} ---")
    ext = Path(file_path).suffix.lower()
    temp_images = []

    # 1) PDF → per-page images
    if ext == ".pdf":
        print("   -> Converting PDF to images (300 dpi)...")
        pages = convert_from_path(file_path, dpi=300)
        for i, p in enumerate(pages, start=1):
            img_path = f"temp_conn_page_{i}.png"
            p.save(img_path, "PNG")
            temp_images.append(img_path)
    else:
        temp_images = [file_path]

    all_connections = []

    # 2) Per-page analysis
    for page_idx, img in enumerate(temp_images, start=1):
        print(f"\n--- Page {page_idx} ---")
        page_data = analyze_image(img)
        conns = page_data.get("connections", [])

        if not isinstance(conns, list):
            print("   -> Unexpected 'connections' format, skipping this page.")
            continue

        if not conns:
            print("   -> Model returned an empty 'connections' list for this page.")
            continue

        for c in conns:
            from_comp = c.get("from_component")
            from_pin  = c.get("from_pin")
            to_comp   = c.get("to_component")
            to_pin    = c.get("to_pin")
            wire_id   = c.get("wire_id")
            wire_gauge= c.get("wire_gauge")
            wire_color= c.get("wire_color")
            core_count= c.get("core_count")
            segments  = c.get("segment_lengths", [])

            # normalize core_count to int if possible
            if core_count is not None and not isinstance(core_count, int):
                try:
                    core_count = int(core_count)
                except Exception:
                    m = re.search(r"\d+", repr(core_count))
                    core_count = int(m.group()) if m else None

            # ensure segments is a list of strings
            if not isinstance(segments, list):
                segments = []
            cleaned_segments = []
            for seg in segments:
                if isinstance(seg, str):
                    s = seg.strip()
                    if s:
                        cleaned_segments.append(s)

            all_connections.append({
                "Page": page_idx,
                "From_Component": from_comp,
                "From_Pin": from_pin,
                "To_Component": to_comp,
                "To_Pin": to_pin,
                "Wire_ID": wire_id,
                "Wire_Gauge": wire_gauge,
                "Wire_Color": wire_color,
                "Core_Count": core_count,
                "Segment_Lengths": cleaned_segments,
            })

    # 3) If nothing, stop
    if not all_connections:
        print("\nNo connections extracted. No Excel created.")
        return

    # 4) Flatten Segment_Lengths into Segment_1..Segment_N
    max_segments = max(len(row.get("Segment_Lengths", [])) for row in all_connections)
    flat_rows = []
    for row in all_connections:
        base = {
            "Page": row["Page"],
            "From_Component": row["From_Component"],
            "From_Pin": row["From_Pin"],
            "To_Component": row["To_Component"],
            "To_Pin": row["To_Pin"],
            "Wire_ID": row["Wire_ID"],
            "Wire_Gauge": row["Wire_Gauge"],
            "Wire_Color": row["Wire_Color"],
            "Core_Count": row["Core_Count"],
        }
        segs = row.get("Segment_Lengths", [])
        for idx in range(max_segments):
            col_name = f"Segment_{idx + 1}"
            base[col_name] = segs[idx] if idx < len(segs) else None
        flat_rows.append(base)

    df_conn = pd.DataFrame(flat_rows)

    excel = f"{Path(file_path).stem}_CONNECTIONS_SEGMENTS.xlsx"
    with pd.ExcelWriter(excel, engine="openpyxl") as writer:
        df_conn.to_excel(writer, sheet_name="Connections", index=False)

    print(f"\n✅ SUCCESS! Saved connections Excel: {excel}")

    # 5) Cleanup temp images
    if ext == ".pdf":
        for t in temp_images:
            Path(t).unlink(missing_ok=True)


print("\nConnection Extractor Ready.")


Connection Extractor Ready (company gateway).

Connection Extractor Ready.


In [25]:
file_path ="Sanitized sheet 1 (var 2).pdf"
process_file(file_path)


--- Processing File: Sanitized sheet 1 (var 2).pdf ---
   -> Converting PDF to images (300 dpi)...

No connections extracted. No Excel created.
